# Telegram Username Checker

1. Edit **Config** cell below
2. Run **Config**, then **Run**
3. On first login — enter the **code from Telegram**

## Config

Run this cell first. Change only the values you need.

In [ ]:
# ── Telegram API (https://my.telegram.org) ──────────────────
API_ID = 12345678
API_HASH = "your_api_hash_here"
PHONE = "+1234567890"
PASSWORD_2FA = "your_2fa_password"

# ── Username generation ─────────────────────────────────────
NICK_COUNT_MIN = 50        # random count range (inclusive)
NICK_COUNT_MAX = 100
LETTER_COUNT = 5           # random chars after @ (Telegram min is 5)
PREFIX = ""                # optional fixed prefix after @

ALLOW_LOWERCASE = True     # a-z
ALLOW_UPPERCASE = True     # A-Z
ALLOW_DIGITS = True        # 0-9
FIRST_CHAR_MUST_BE_LETTER = True # tg wants the first character after @ to be a letter

# ── Output files ────────────────────────────────────────────
INPUT_FILE = "Input.txt" # the file with usernames
AVAILABLE_FILE = "Available.txt" # the file with available usernames
OVERWRITE_INPUT = True # overwrite the input file?
APPEND_AVAILABLE = False # False = overwrite Available.txt with free usernames from this run only, True  = append free usernames from this run to Available.txt

# ── Checking ────────────────────────────────────────────────
CHECK_DELAY_SEC = 1

# ── Session ─────────────────────────────────────────────────
SESSION_NAME = "telegram"

print("Config loaded.")

## Run

Run this cell after Config.

In [ ]:
import asyncio
import os
import random
import string
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "telethon"])

from telethon import TelegramClient
from telethon.errors import (
    FloodWaitError,
    SessionPasswordNeededError,
    UsernameInvalidError,
    UsernamePurchaseAvailableError,
)
from telethon.tl.functions.account import CheckUsernameRequest


def build_charset() -> str:
    charset = ""
    if ALLOW_LOWERCASE:
        charset += string.ascii_lowercase
    if ALLOW_UPPERCASE:
        charset += string.ascii_uppercase
    if ALLOW_DIGITS:
        charset += string.digits
    if not charset:
        raise ValueError("Enable at least one of ALLOW_LOWERCASE, ALLOW_UPPERCASE, ALLOW_DIGITS")
    if FIRST_CHAR_MUST_BE_LETTER and not any(c.isalpha() for c in charset):
        raise ValueError("FIRST_CHAR_MUST_BE_LETTER requires letters in charset")
    return charset


def generate_username(charset: str) -> str:
    letters = "".join(c for c in charset if c.isalpha())
    while True:
        random_part = "".join(random.choice(charset) for _ in range(LETTER_COUNT))
        body = f"{PREFIX}{random_part}"
        if FIRST_CHAR_MUST_BE_LETTER and body and not body[0].isalpha():
            body = random.choice(letters) + body[1:]
        if len(body) >= 5:
            return f"@{body}"


charset = build_charset()
SESSION_FILE = f"{SESSION_NAME}.session"
NICK_COUNT = random.randint(NICK_COUNT_MIN, NICK_COUNT_MAX)

print("[1/4] Connecting to Telegram...")
client = TelegramClient(SESSION_NAME, API_ID, API_HASH)
await client.connect()

if not await client.is_user_authorized():
    await client.send_code_request(PHONE)
    code = input("Enter code from Telegram: ")
    try:
        await client.sign_in(PHONE, code)
    except SessionPasswordNeededError:
        await client.sign_in(password=PASSWORD_2FA)

me = await client.get_me()
print(f"OK: {me.first_name} | id={me.id}")

print(f"[2/4] Generating {NICK_COUNT} usernames (random {NICK_COUNT_MIN}-{NICK_COUNT_MAX})...")
print(f"    format: @{PREFIX}{'?' * LETTER_COUNT}")
print(f"    charset: lowercase={ALLOW_LOWERCASE}, uppercase={ALLOW_UPPERCASE}, digits={ALLOW_DIGITS}")

usernames = set()
while len(usernames) < NICK_COUNT:
    usernames.add(generate_username(charset))
usernames = sorted(usernames)

input_mode = "w" if OVERWRITE_INPUT else "a"
with open(INPUT_FILE, input_mode, encoding="utf-8") as f:
    if input_mode == "a" and os.path.exists(INPUT_FILE) and os.path.getsize(INPUT_FILE) > 0:
        f.write("\n")
    f.write("\n".join(usernames) + "\n")
print("Examples:", ", ".join(usernames[:5]))

print("[3/4] Checking availability...")
available = []
for i, username in enumerate(usernames, 1):
    name = username.lstrip("@")
    try:
        is_free = await client(CheckUsernameRequest(username=name))
        status = "AVAILABLE" if is_free else "taken"
        if is_free:
            available.append(username)
    except UsernameInvalidError:
        status = "invalid"
    except UsernamePurchaseAvailableError:
        status = "auction"
    except FloodWaitError as e:
        await asyncio.sleep(e.seconds)
        is_free = await client(CheckUsernameRequest(username=name))
        status = "AVAILABLE" if is_free else "taken"
        if is_free:
            available.append(username)
    print(f"[{i}/{NICK_COUNT}] @{name} — {status}")
    await asyncio.sleep(CHECK_DELAY_SEC)

available_mode = "a" if APPEND_AVAILABLE else "w"
with open(AVAILABLE_FILE, available_mode, encoding="utf-8") as f:
    if available_mode == "a" and os.path.exists(AVAILABLE_FILE) and os.path.getsize(AVAILABLE_FILE) > 0:
        f.write("\n")
    f.write("\n".join(available) + ("\n" if available else ""))

await client.disconnect()

print(f"[4/4] Done! Available: {len(available)}")
print(f"  {INPUT_FILE} — all generated usernames")
print(f"  {AVAILABLE_FILE} — available usernames")
print(f"  {SESSION_FILE} — session file (keep for next run)")